In [77]:
import pandas as pd
import networkx as nx
import os
from dotenv import load_dotenv, find_dotenv
import igraph as ig

In [78]:
load_dotenv(find_dotenv())
root = os.getenv("ROOT_DIR")

graph_df = pd.read_csv(os.path.join(root, 'data', 'cleaned_collaboration.csv'))

In [79]:
print(graph_df.shape)
print(graph_df.columns)

(188305, 14)
Index(['id_1', 'id_2', 'name_1', 'followers_1', 'popularity_1', 'genres_1',
       'chart_hits_1', 'top_chart_1', 'name_2', 'followers_2', 'popularity_2',
       'genres_2', 'chart_hits_2', 'top_chart_2'],
      dtype='object')


In [80]:
import math
import random
import pandas as pd
import igraph as ig
import numpy as np
from tqdm import tqdm

# Correct import and library for ForceAtlas2
import networkx as nx
from pyforceatlas2 import ForceAtlas2


def build_graph(df: pd.DataFrame, root: str = ".", spread_radius: float = 4.0, z_randomness: float = 0.3, subset: float = 1):
    """
    Build graph with optional parameters:
    - spread_radius: multiplier for x,y coordinate spread (default 4.0 for more spread)
    - z_randomness: fraction of base z used as randomness (default 0.3)
    - subset: fraction of data to use (e.g., 0.1 for 10%) for faster testing. Default is None (full dataset).

    Z mapping (popularity → z, heavy-tailed):
      • Bottom 50% by popularity  → z ∈ [0, 100]  (clamped to this band)
      • Top    50% by popularity  → z ∈ [100, 1500] with concave mapping to keep most near 100
    """
    # ---- NEW: Handle the subset parameter ----
    if subset is not None and 0 < subset < 1.0:
        print(f"--- Using a random subset of {subset:.0%} of the data ---")
        df = df.sample(frac=subset, random_state=42).reset_index(drop=True)
    # -----------------------------------------

    # 1) collect unique node attributes
    node_attrs, dropped_idx = {}, set()
    for side in (1, 2):
        id_col = f"id_{side}"
        cols   = {k: f"{k}_{side}" for k in
                  ("name", "followers", "popularity",
                   "genres", "chart_hits", "top_chart")}

        for idx, row in tqdm(df.iterrows(), total=len(df),
                             desc=f"Collect attrs side {side}", unit="row"):
            if idx in dropped_idx:
                continue
            nid   = row[id_col]
            attrs = {k: row[v] for k, v in cols.items()}
            if nid in node_attrs:
                for k, v in attrs.items():
                    if k in ("chart_hits", "top_chart"):
                        continue
                    if node_attrs[nid][k] != v:
                        dropped_idx.add(idx)
                        break
            else:
                node_attrs[nid] = attrs

    # 2) build edge list (skip conflicting rows)
    ids    = list(node_attrs)
    id2idx = {nid: i for i, nid in enumerate(ids)}
    edges  = []
    for i, row in tqdm(df.iterrows(), total=len(df),
                       desc="Building edges", unit="row"):
        if i in dropped_idx:
            continue
        # Ensure both nodes from the edge are in our collected attributes before adding
        if row.id_1 in id2idx and row.id_2 in id2idx:
            edges.append((id2idx[row.id_1], id2idx[row.id_2]))

    # 3) create igraph
    g = ig.Graph(n=len(ids), edges=edges, directed=False)
    for attr in ("name", "followers", "popularity",
                 "genres", "chart_hits", "top_chart"):
        g.vs[attr] = [node_attrs[n][attr] for n in ids]
    g.vs["degree"] = g.degree()

    # 3.5) community detection (Louvain)
    comms = g.community_multilevel()
    g.vs["community"] = comms.membership

    # 4) compute layout
    main_ids = [v.index for v in g.vs if v["degree"] > 0]
    iso_ids  = [v.index for v in g.vs if v["degree"] == 0]
    coords   = [[math.nan, math.nan] for _ in range(g.vcount())]

    if main_ids:
        sub = g.induced_subgraph(main_ids)

        # 1. Convert igraph subgraph to a networkx graph.
        nx_graph = nx.Graph()
        nx_graph.add_nodes_from(range(sub.vcount()))
        nx_graph.add_edges_from(sub.get_edgelist())

        # 2. Instantiate the ForceAtlas2 class with valid settings.
        forceatlas2_layout = ForceAtlas2(
            gravity=1.0,
            verbose=False
        )

        # 3. Run the layout algorithm by calling the method on the instance.
        positions = forceatlas2_layout.forceatlas2_networkx_layout(
            graph=nx_graph,
            pos=None,
            iterations=30
        )
        
        # 4. Convert positions dict back to a list for igraph compatibility.
        lay = [positions[i] for i in range(sub.vcount())]

        for loc, glob in enumerate(main_ids):
            coords[glob] = lay[loc]

    if iso_ids:
        r = max((max(abs(x), abs(y)) for x, y in coords if not math.isnan(x)), default=1.0)
        for k, vid in tqdm(enumerate(iso_ids), total=len(iso_ids),
                           desc="Positioning isolates", unit="iso"):
            phi = 2 * math.pi * k / max(len(iso_ids), 1)
            coords[vid] = (r * 1.05 * math.cos(phi), r * 1.05 * math.sin(phi))

    # 5) normalize x, y with spread
    xs = [c[0] for c in coords]
    ys = [c[1] for c in coords]
    x_min, x_max = (min(xs), max(xs)) if xs else (0.0, 1.0)
    y_min, y_max = (min(ys), max(ys)) if ys else (0.0, 1.0)

    def normalize_with_spread(v, vmin, vmax, spread):
        base_range = 5000
        expanded_range = base_range * spread
        if math.isnan(v) or vmax <= vmin:
            return expanded_range * 0.5
        return expanded_range * (v - vmin) / (vmax - vmin)

    xs_norm = [normalize_with_spread(x, x_min, x_max, spread_radius) for x in xs]
    ys_norm = [normalize_with_spread(y, y_min, y_max, spread_radius) for y in ys]

    # z from popularity with percentile banding
    pops = pd.to_numeric(pd.Series(g.vs["popularity"]), errors="coerce").fillna(0.0).to_numpy(float)

    if pops.size == 0:
        zs = [0.0] * g.vcount()
    else:
        p_min = float(np.min(pops))
        p_max = float(np.max(pops))
        p_med = float(np.quantile(pops, 0.5))

        low_denom  = max(p_med - p_min, 1e-12)
        high_denom = max(p_max - p_med, 1e-12)
        
        z_low_min,  z_low_max  = 0.0, 300.0
        z_high_min, z_high_max = 500.0, 3000.0
        gamma_high = 2.2

        zs = []
        for pop in pops:
            if pop <= p_med or high_denom <= 1e-12:
                r = (pop - p_min) / low_denom
                z_base = z_low_min + r * (z_low_max - z_low_min)
                mag = max(z_base, 4.0)
                jitter = random.uniform(-z_randomness, z_randomness) * mag
                z = max(z_low_min, min(z_low_max, z_base + jitter))
            else:
                r = (pop - p_med) / high_denom
                z_base = z_high_min + (r ** gamma_high) * (z_high_max - z_high_min)
                mag = max(z_base, 8.0)
                jitter = random.uniform(-z_randomness, z_randomness) * mag
                z = max(z_high_min, min(z_high_max, z_base + jitter))
            zs.append(z)

    # Final DataFrame creation
    degrees = g.vs["degree"]
    communities = g.vs["community"]

    nodes_df = (
        pd.DataFrame.from_dict(node_attrs, orient="index")
          .reset_index().rename(columns={"index": "id"})
          .assign(x=xs_norm, y=ys_norm, z=zs,
                  degree=degrees, community=communities, popularity=pops)
    )

    edges_df = (
        pd.DataFrame(edges, columns=["src_idx", "tgt_idx"])
          .assign(source=lambda d: d.src_idx.map(lambda i: ids[i]),
                  target=lambda d: d.tgt_idx.map(lambda i: ids[i]))
          .loc[:, ["source", "target"]]
    )
    
    return g, nodes_df, edges_df

In [ ]:
g, nodes, edges = build_graph(graph_df)



Building edges: 100%|██████████| 188305/188305 [00:09<00:00, 19602.53row/s]


In [ ]:
# g.to_csv(os.path.join(root, 'data', 'constructed_network.csv'))

In [ ]:
nodes.to_csv(os.path.join(root, 'data', 'network_nodes.csv'))
edges.to_csv(os.path.join(root, 'data', 'network_edges.csv'))